# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [1]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Execution time of {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Task completed.")

example_task()

Task completed.
Execution time of example_task: 0.5052 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [2]:
import functools

def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Check positional arguments for lists
        for arg in args:
            if isinstance(arg, list):
                print(f"List length in positional argument: {len(arg)}")
        # Check keyword arguments for lists
        for key, value in kwargs.items():
            if isinstance(value, list):
                print(f"List length in '{key}': {len(value)}")
        return func(*args, **kwargs)
    return wrapper

@show_list_length
def process_data(data_list, name):
    print(f"Processing {name}")

process_data([10, 20, 30], "Test Data")

List length in positional argument: 3
Processing Test Data


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [3]:
import functools
from datetime import datetime
import time

def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start_time = time.perf_counter()
            result = func(*args, **kwargs)
            end_time = time.perf_counter()
            duration = end_time - start_time
            
            # Append logs to file with timestamp
            with open(filename, "a", encoding="utf-8") as f:
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                f.write(f"[{timestamp}] Function: {func.__name__}, Duration: {duration:.4f}s\n")
            
            return result
        return wrapper
    return decorator

@logger("lab1.log")
def some_slow_function():
    time.sleep(0.5)
    print("Function execution complete.")

some_slow_function()

Function execution complete.


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [4]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Invalid email format: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, first_name, last_name, email):
        self.first_name = first_name
        self.last_name = last_name
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Student created: {s.email}")
except ValueError as e:
    print(e)

Student created: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [5]:
class AccessLogger:
    def __init__(self):
        self._name = None

    def __set_name__(self, owner, name):
        self._name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = instance.__dict__.get(self._name)
        print(f"[GET LOG] Accessing '{self._name}': {value}")
        return value

    def __set__(self, instance, value):
        print(f"[SET LOG] Updating '{self._name}' to: {value}")
        instance.__dict__[self._name] = value

class User:
    name = AccessLogger()
    age = AccessLogger()

    def __init__(self, name, age):
        self.name = name
        self.age = age

# Test:
user = User("Dominik", 25)
print(f"User name: {user.name}")
user.age = 26

[SET LOG] Updating 'name' to: Dominik
[SET LOG] Updating 'age' to: 25
[GET LOG] Accessing 'name': Dominik
User name: Dominik
[SET LOG] Updating 'age' to: 26


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [6]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [7]:
def collatz_generator(n):
    while n != 1:
        yield n
        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1
    yield 1  # Always end at 1

# Test:
for step in collatz_generator(10):
    print(f"Step: {step}")

Step: 10
Step: 5
Step: 16
Step: 8
Step: 4
Step: 2
Step: 1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [8]:
current_user = {"username": "admin", "role": "superuser"}

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError(f"Access Denied! Required role: {role}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_role("superuser")
def secure_action():
    print("Admin action executed successfully.")

secure_action()

Admin action executed successfully.


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [9]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type
        self._name = None

    def __set_name__(self, owner, name):
        self._name = name

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"Attribute '{self._name}' must be of type {self.expected_type.__name__}")
        instance.__dict__[self._name] = value

class Profile:
    age = Typed(int)
    name = Typed(str)

profile = Profile()
profile.age = 30  # OK
# profile.name = 123  # Would raise TypeError

### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [10]:
def prime_generator():
    num = 2
    while True:
        if all(num % i != 0 for i in range(2, int(num**0.5) + 1)):
            yield num
        num += 1

# Generator expression for primes ending in 7
primes_ending_7 = (p for p in prime_generator() if str(p).endswith('7'))

# Print first 5 primes ending in 7
for _ in range(5):
    print(f"Prime ending in 7: {next(primes_ending_7)}")

Prime ending in 7: 7
Prime ending in 7: 17
Prime ending in 7: 37
Prime ending in 7: 47
Prime ending in 7: 67
